In [ ]:
# importing functions
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as ss
#%matplotlib ipympl

In [ ]:
# extracting the gamma passing rates (%GP) for each structure from the Pinnacle and RayStation scans (prostate scans)
#NOTE the following paths is where the results and gpr data files are in your PC. They must be csv (comma separated) or txt files.
# the data with the gamma passing rates (gpr) are the structure_df files made in the other python notebook
# the data for the results are in the results_df files made in the GI_local_gamma_notebook.ipynb

rs_pros_results_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\33pats_rs_pros_results_df.txt"
rs_pros_gpr_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\33pats_rs_pros_structures_df.txt"

pn_pros_results_path = r'C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\30_pn_pros_results_df.csv'
pn_pros_gpr_path = r'C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\30_pn_pros_structures_df.csv'

rs_rect_results_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\16pats_rs_rect_results_df.txt"
rs_rect_gpr_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\16pats_rs_rect_structures_df.txt"

pn_rect_results_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\27pats_pn_rect_results_df.txt"
pn_rect_gpr_path = r"C:\Users\slaar\Desktop\project\pinnacle_RS_comparison\27pats_pn_rect_structures_df.txt"

#NOTE the folder you want to save your plots and mann whitney u test tables
pros_save_path = 'C:\\Users\\slaar\\Desktop\\project\\pinnacle_RS_comparison\\prostate_plots'
rect_save_path = 'C:\\Users\\slaar\\Desktop\\project\\pinnacle_RS_comparison\\rectum_plots'

def get_gpr(path):
    '''returns the gamma passing rate information from the csv or txt files'''
    if path[-3:] == 'csv':
        data = np.genfromtxt(path, delimiter = ',', dtype = None)
    if path[-3:] == 'txt':
        data = np.genfromtxt(path, delimiter = '\t', dtype = None)
    return data


def get_results(path):
    '''returns the action and tolerance limits from the csv or txt files'''
    if path[-3:] == 'csv':
        data = np.genfromtxt(path, delimiter = ',', dtype = None, skip_header = 1)
    if path[-3:] == 'txt':
        data = np.genfromtxt(path, delimiter = '\t', dtype = None, skip_header = 1)
    return data



#NOTE that raystation data and ps_rect is stored in a .txt file and pinnacle prostate data is stored in a .csv file: the delimiter for the pinnacle data is a comma, while the delimiter for the raystation data is a tab
#NOTE that the pinnacle prostate files were created by the previous notebook, GI_local_gamma_notebook.ipynb. The pinnacle rectum and Raystation data was created on the remote desktop and stored in the one drive and then copied into a text file.


In [ ]:
def get_dict (gprs,tl_al):
    '''create a dictionary combining the gprs and tl&als for a specific type of scan ie raystation rectum'''
    dictionary = {}
    for index in range(0, len(gprs[0])):
        list_gprs = []
        for i in range(1,(len(gprs))):
            list_gprs.append(float(gprs[i][index]))
        list_tl = float(tl_al[index][1])
        list_al = float(tl_al[index][2])
        dictionary[gprs[0][index]] = [list_gprs, list_tl, list_al]
    return dictionary



In [ ]:
def all_names (dictionary):
    '''requires the dictionary and returns a list of all the structure names (list of all the keys)'''
    return [*dictionary]


### Plotting

In [ ]:
def histogram (dictionary, pros_or_rect, pn_or_rs):
    '''Creates the histograms for the gamma passing rates. 
    pros_or_rect and pn_or_rs must be strings
    '''
    for i in all_names(dictionary):
        savefile1 = i.replace('.', '')
        savefile11 = savefile1.replace('Merlin Structure: ', '')
        savefile12 = savefile11.replace('+', '')
        savefile2 = savefile12.replace(' ', '')
        savefile3 = savefile2.replace('(', '')
        savefile4 = savefile3.replace(')', '')
        savefile5 = savefile4.replace(r'/', '')
        savefile6 = savefile5.replace('%', '')
        mean = np.mean(dictionary[i][0])
        std = np.std(dictionary[i][0], ddof = 1)
        median = np.median(dictionary[i][0])
        plt.figure(figsize = (9,5))
        plt.title(f'Gamma Passing rates for the structure - {i}')
        y,x,_ = plt.hist(dictionary[i][0], color = 'grey', label = r'%GP Entries', bins = 20, edgecolor = 'black')
        plt.ylim(0, y.max()+5)
        plt.vlines(mean, color = 'black', label = f'Mean = {np.round(mean, 3)} $\pm$ {np.round(std,3)}', ymin = 0, ymax = y.max()+5, linewidth = 2, linestyle = 'dashed')
        plt.vlines(median, color = 'navy', label = f'Median = {np.round(median, 3)}', ymin = 0, ymax = y.max()+5, linewidth = 2, linestyle = 'dashed')
        plt.vlines(dictionary[i][1], color = 'green', label = f'Tolerance Limit: {dictionary[i][1]}', ymin = 0, ymax = y.max()+5, linewidth = 2, linestyle = 'dotted')
        plt.vlines(dictionary[i][2], color = 'red', label = f'Action Limit: {dictionary[i][2]}', ymin = 0, ymax = y.max()+5, linewidth = 2, linestyle = 'dotted')
        plt.grid()
        plt.legend(loc = 'upper center', fontsize = 10)
        plt.tight_layout()
        if pn_or_rs == 'pn':
            savefile7 = savefile6 + '_pn'
        if pn_or_rs == 'rs':
            savefile7 = savefile6 + '_rs'
        if pros_or_rect == 'pros':
            savefile = savefile7 + '_pros'
            plt.savefig(f'{pros_save_path}\\{savefile}.pdf')
        if pros_or_rect == 'rect':
            savefile = savefile7 + '_rect'
            plt.savefig(f'{rect_save_path}\\{savefile}.pdf')



### Statistics: Mann Whitney U Test

In [ ]:
def mann_whit_utest (pn_dict, rs_dict, pros_or_rect):
    '''returns the mann whitnet u test, which compares the gamma passing rates for each treatment planning system.
    The if statements are for the discrepancies in the naming of the structures
    The 'IDC ROI (3%/2mm).' structure is written as 'IDC ROI(3%/2mm).' only for rs_rect'''
    all_utest_dict = {}
    all_utest_arr = [['structure', 'utest', 'pval']]
    for i in all_names(rs_dict):
        for k in all_names(pn_dict):
            if i==k:
                utest, pval = ss.mannwhitneyu(pn_dict[i][0], rs_dict[i][0], alternative = 'two-sided')
                all_utest_dict[i] = utest
                all_utest_arr.append([i,utest,pval])

            elif i == np.str_('IDC ROI (3%/3mm).') and k == np.str_('Merlin Structure: PTVp+OAR DoseCheck20 (3%/3mm).'):
                utest, pval = ss.mannwhitneyu(pn_dict[k][0], rs_dict[i][0], alternative = 'two-sided')
                all_utest_dict[np.str_('Independent Dose Checks (3%/3mm).')] = utest
                all_utest_arr.append([np.str_('Independent Dose Checks (3%/3mm).'),utest,pval])

            elif i == np.str_('IDC ROI(3%/2mm).') and k == np.str_('Merlin Structure: PTVp+OAR DoseCheck20(3%/2mm).'):
                utest, pval = ss.mannwhitneyu(pn_dict[k][0], rs_dict[i][0], alternative = 'two-sided')
                all_utest_dict[np.str_('Independent Dose Checks (3%/2mm).')] = utest
                all_utest_arr.append([np.str_('Independent Dose Checks (3%/2mm).'),utest,pval])
            elif i == np.str_('IDC ROI(3%/2mm).') and k == np.str_('IDC ROI (3%/2mm).'):
                utest, pval = ss.mannwhitneyu(pn_dict[k][0], rs_dict[i][0], alternative = 'two-sided')
                all_utest_dict[np.str_('IDC ROI (3%/2mm).')] = utest
                all_utest_arr.append([np.str_('IDC ROI (3%/2mm).'),utest,pval])
            else:
                continue 
    all_utest_df = pd.DataFrame(all_utest_arr, columns=['Structure', 'Utest', 'P-values'])
    if pros_or_rect == 'pros':
        all_utest_df.to_csv(f'{pros_save_path}\\mannwhitneyu_test_results.csv', index = False)
    if pros_or_rect == 'rect':
        all_utest_df.to_csv(f'{rect_save_path}\\mannwhitneyu_test_results.csv', index = False)
    return all_utest_df

In [ ]:
#NOTE Using the dictionaries

pn_pros_gpr = get_gpr(pn_pros_gpr_path)
pn_pros_tl_al = get_results(pn_pros_results_path)

rs_pros_gpr = get_gpr(rs_pros_gpr_path)
rs_pros_tl_al = get_results(rs_pros_results_path)

pn_rect_gpr = get_gpr(pn_rect_gpr_path)
pn_rect_tl_al = get_results(pn_rect_results_path)

rs_rect_gpr = get_gpr(pn_rect_gpr_path)
rs_rect_tl_al = get_results(pn_rect_results_path)

pn_pros_dict = get_dict(get_gpr(pn_pros_gpr_path),get_results(pn_pros_results_path))
pn_rect_dict = get_dict(get_gpr(pn_rect_gpr_path),get_results(pn_rect_results_path))
rs_pros_dict = get_dict(get_gpr(rs_pros_gpr_path),get_results(rs_pros_results_path))
rs_rect_dict = get_dict(get_gpr(rs_rect_gpr_path),get_results(rs_rect_results_path))


histogram(pn_pros_dict, 'pros', 'pn')
histogram(rs_pros_dict, 'pros', 'rs')
histogram(pn_rect_dict, 'rect', 'pn')
histogram(rs_rect_dict, 'rect', 'rs')


# mann_whit_utest(pn_pros_dict, rs_pros_dict, 'pros')

In [ ]:
mann_whit_utest(pn_pros_dict, rs_pros_dict, 'pros')


In [ ]:
mann_whit_utest(pn_rect_dict, rs_rect_dict, 'rect')
